# x3c_infracoes — Geocodificação e Concordância Territorial de Autos IBAMA

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** correspondência espacial de autos (IBAMA), registros individuais, municipal, ano da lavratura  
**Conceito e Mecanismo Econômico:** Auditoria de correspondência geográfica entre o município declarado pelo autuado/fiscal e o município resultante da sobreposição espacial das coordenadas.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__autos_ibama__geocodificacao_candidata.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar divergências entre codigo_ibge e codigo_ibge_espacial; subconjunto com coordenadas.
> - Não constitui uma quarta medida independente de fiscalização.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`SEQ_AUTO_INFRACAO`**: ID do auto.
- **`codigo_ibge`**: Município declarado.
- **`codigo_ibge_espacial`**: Município determinado pelas coordenadas.
- **`diverge_codigo_informado`**: Booleano indicando divergência territorial.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__autos_ibama__geocodificacao_candidata.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x3c_infracoes`:
- **Atenção:** Explorar divergências entre codigo_ibge e codigo_ibge_espacial; subconjunto com coordenadas.
- **Atenção:** Não constitui uma quarta medida independente de fiscalização.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df['diverge_codigo_informado'] = df['diverge_codigo_informado'].astype(bool)
# REGRA DO TXT: 'Explorar divergências entre codigo_ibge e codigo_ibge_espacial; subconjunto com coordenadas.'
taxa = df['diverge_codigo_informado'].mean()
print(f"Taxa global de divergência territorial: {taxa:.2%}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Contagem de divergências:")
print(df['diverge_codigo_informado'].value_counts())

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
df['diverge_codigo_informado'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['mediumseagreen', 'coral'])
plt.title("Proporção de Autos com Divergência Territorial")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
div_ano = df.groupby('ano')['diverge_codigo_informado'].mean().reset_index()
sns.lineplot(data=div_ano, x='ano', y='diverge_codigo_informado', marker='o')
plt.title("Taxa de Divergência Territorial por Ano")
plt.ylabel("Fração Divergente")
plt.show()

## Fase 6: Feature Engineering — Geocodificação e Consistência Espacial

### 1. Step 1: Validação de Correspondência Territorial
- Comparamos o código municipal declarado pelo fiscal no auto de infração (`codigo_ibge`) com o código obtido pela sobreposição espacial das coordenadas geográficas (`codigo_ibge_espacial`).
- Identificamos discrepâncias causadas por erros de digitação ou imprecisão de GPS.

### 2. Step 2: Indicador de Precisão Cartográfica
- Calculamos a taxa de concordância municipal por ano, servindo como variável de controle de qualidade dos dados de fiscalização.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Taxa de concordância espacial por município
conf_mun = df.groupby('codigo_ibge')['diverge_codigo_informado'].agg(
    total_autos=('count'),
    taxa_divergencia=('mean')
).reset_index()

conf_mun['taxa_concordancia'] = 1.0 - conf_mun['taxa_divergencia']

print("Qualidade da Geocodificação dos Autos do Ibama:")
print(conf_mun.describe())
print(conf_mun.head())